# Water Balance for Canopy-enabled simulations

This sheet checks and documents the water balance for ATS simulations based on models such as "priestly_taylor_canopy_evapotranspiration" that include four water pools -- canopy, snow, surface water, and subsurface water.  We can verify each of these independently.

Note all of these are done in units of [m] or [m/s], which means dividing by surface area or molar density as required to convert to these units.


In [ ]:
dirname = './SF01-0-updateLAItime'
#dirname = 'priestley_taylor_canopy_evapotranspiration_relperm_trf_surftemp.demo'
cv_key = 'canopy-cell_volume'

In [ ]:
import h5py
import pandas
import numpy as np
from matplotlib import pyplot as plt
import sys, os

In [ ]:
with h5py.File(os.path.join(dirname, 'ats_vis_surface_data.h5'),'r') as d:
    a_key = list(d[cv_key].keys())[0]
    surf_area = d[cv_key][a_key][:].sum() # m^2

In [ ]:
# load data
df = pandas.read_csv(os.path.join(dirname, 'water_balance.dat'), comment='#')
time = df['time [d]']


In [ ]:
df

In [ ]:
df['rain precipitation [m d^-1]']

In [ ]:
df['rain precipitation [m d^-1]']

fig, ax = plt.subplots(1,1, figsize=(10,3))

ax.plot(time, df['rain precipitation [m d^-1]'], 'b', label='rain')
ax.plot(time, df['snow precipitation [m d^-1]'], 'c', label='snow')

ax.set_ylabel('precip [m s^-1]')
ax.legend()

In [ ]:
# process -- global water balance -- converting all to m/day
gf = dict()
gf['P'] = df['rain precipitation [m d^-1]']
gf['S'] = df['snow precipitation [m d^-1]']
gf['ET'] = df['evapotranspiration [m d^-1]']
gf['Q'] = df['runoff generation [mol d^-1]']/55000./surf_area
gf['F_lf'] = -df['left face flux']/55000./surf_area #efflux
gf['F_rf'] = df['right face flux']/55000./surf_area # efflux

gf_net = gf['P'] + gf['S'] - gf['ET'] - gf['Q'] + gf['F_lf'] - gf['F_rf']
global_water = (df['canopy water content [mol]'] + df['snow water content [mol]'] + df['surface water content [mol]'] + df['subsurface water content [mol]']) / 55500 / surf_area


In [ ]:
import modvis.ats_xdmf as xdmf
from modvis import colors

In [ ]:
import h5py

model_dir = "./SF01-0-updateLAItime"
cv_key = 'surface-cell_volume'

generate_plots = False

xlim_preset = (-10,690)
ylim_preset = (940,1120)
zlim_preset = (0.0, 1.0)

In [ ]:
vis_surf = xdmf.VisFile(directory=model_dir, prefix='ats_vis',
                            domain="surface", 
                            # filename="ats_vis_surface_data.h5" , 
                            # mesh_filename="ats_vis_surface_mesh.h5", 
                            model_time_unit='d')
#select same output as subsurface
#vis_surf.filterIndices(steps)
vis_surf.loadMesh(order=['x','z'])

vis = xdmf.VisFile(directory=model_dir, prefix='ats_vis',
                       # filename="ats_vis_data.h5", 
                       # mesh_filename="ats_vis_mesh.h5", 
                       model_time_unit='d')
#select output every 2 days
#vis.filterIndices(steps)
vis.loadMeshPolygons()

In [ ]:
# [to do] 250522
# with the update of "1-full_workflow_OakCreek.ipynb"
# rightx/righty/22.0 below can be loaded from "../data-processed/SF01/m2_SF01_nx100.mat"

# topx and topy are from 1-full_workflow_OakCreek.ipynb
rightx = np.array([  0. ,   6.8,  13.6,  20.4,  27.2,  34. ,  40.8,  47.6,  54.4,  61.2,
  68. ,  74.8,  81.6,  88.4,  95.2, 102. , 108.8, 115.6, 122.4, 129.2,
 136. , 142.8, 149.6, 156.4, 163.2, 170. , 176.8, 183.6, 190.4, 197.2,
 204. , 210.8, 217.6, 224.4, 231.2, 238. , 244.8, 251.6, 258.4, 265.2,
 272. , 278.8, 285.6, 292.4, 299.2, 306. , 312.8, 319.6, 326.4, 333.2,
 340. , 346.8, 353.6, 360.4, 367.2, 374. , 380.8, 387.6, 394.4, 401.2,
 408. , 414.8, 421.6, 428.4, 435.2, 442. , 448.8, 455.6, 462.4, 469.2,
 476. , 482.8, 489.6, 496.4, 503.2, 510. , 516.8, 523.6, 530.4, 537.2,
 544. , 550.8, 557.6, 564.4, 571.2, 578. , 584.8, 591.6, 598.4, 605.2,
 612. , 618.8, 625.6, 632.4, 639.2, 646. , 652.8, 659.6, 666.4, 673.2,
 680. ])
righty = np.array([1102.011, 1101.917, 1101.917, 1101.917, 1099.421, 1099.421, 1098.469,
 1097.036, 1090.794, 1080.269, 1077.915, 1075.28 , 1072.323, 1072.323,
 1065.303, 1061.85 , 1057.017, 1055.526, 1053.547, 1053.547, 1053.547,
 1053.547, 1053.547, 1053.547, 1053.547, 1053.547, 1053.547, 1051.923,
 1051.923, 1051.923, 1051.923, 1051.923, 1051.499, 1051.009, 1048.841,
 1048.111, 1046.253, 1043.301, 1042.374, 1041.46 , 1041.46 , 1039.801,
 1039.101, 1036.813, 1036.478, 1035.245, 1034.981, 1033.532, 1033.532,
 1033.099, 1032.963, 1032.855, 1032.711, 1031.332, 1030.189, 1030.11 ,
 1029.083, 1029.083, 1028.996, 1028.902, 1028.769, 1028.612, 1027.619,
 1026.707, 1026.257, 1026.036, 1026.036, 1024.497, 1024.008, 1023.479,
 1022.879, 1022.248, 1017.741, 1016.066, 1015.267, 1015.267, 1012.226,
 1011.418, 1010.598, 1009.746, 1004.037, 1001.763, 1001.763, 1001.763,
 1001.763, 1001.763,  999.904,  999.904,  999.476,  993.245,  992.59 ,
  991.7  ,  991.7  ,  990.476,  988.979,  987.216,  983.303,  981.041,
  976.897,  972.046,  972.046])
leftx=rightx
lefty=righty-22.0 # due toin 1-full_workflow_OakCreek.ipynb, extrude 22m in depth

In [ ]:
# plot Flow Path along the 2D Transect
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib import path as mpath
from matplotlib import patches as mpatches
import numpy as np
import matplotlib.colors as colors

from matplotlib.ticker import FixedLocator,FormatStrFormatter

In [ ]:
def create_tricontour_plot(step, vis,varn, log, clim, 
             cmap='Red', xlim=None, ylim=None, showtime=True, ax=None):
    # Get the x, y, and z data for the tricontour plot
    xdata = vis.centroids[:, 0]
    ydata = vis.centroids[:, 2]
    zdata = vis.get(varn, vis.cycles[step])# can do unit conversion here, *32*1000*55
    if log: zdata = np.log10(zdata)
    #if np.any(zdata>7.5):
        # Add a small positive value to zdata to avoid zeros or negative values
     #   zdata[zdata>7.5] = 7.5

    vertices = np.vstack([np.concatenate([leftx, rightx[::-1]]), np.concatenate([lefty, righty[::-1]])]).T
    poly_codes = [mpath.Path.MOVETO] + (len(vertices) - 1) * [mpath.Path.LINETO]
    path = mpath.Path(vertices, poly_codes)
    clip_patch = mpatches.PathPatch(path, facecolor='none', edgecolor='none')
    ax.add_patch(clip_patch)

    # Create the tricontour plot with the current clipping patch
    # the user can changed the last number to get different contour like from 10 to 100
    bounds=np.linspace(zlim_preset[0],zlim_preset[1]*1.001,100)
    cont = ax.tricontourf(xdata, ydata, zdata, vmax=zlim_preset[1], vmin=zlim_preset[0],levels=bounds, cmap=cmap)

    for col in cont.collections:
        col.set_clip_path(clip_patch.get_path(), clip_patch.get_transform())
    

    #cont.set_clim(*clim) # set colorbar limits
    cbar=fig.colorbar(cont, ax=ax, label='Saturation', pad=0.02)
    #cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))  # Set the decimal places
    # Set the tick locations to the minimum and maximum values
    cbar.ax.yaxis.set_major_locator(FixedLocator([zlim_preset[0],0.25,0.50,0.75, zlim_preset[1]]))
    # Format the tick labels with sufficient decimal places
    cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f')) # Adjust as needed
    # Optionally, you can remove intermediate ticks if any are present
    cbar.ax.yaxis.set_minor_locator(FixedLocator([0.25,0.50,0.75]))
    # Control colorbar tick label font size
    cbar.ax.tick_params(labelsize=10) # Adjust 10 as needed

    #fig.colorbar(axes[0].collections[0], ax=axes, label='[Log(mol/L)]')
    ax.set_xlim(xlim_preset)
    ax.set_xticks(np.arange(0, 680, 100))
    ax.set_ylim(ylim_preset)
    ax.set_yticks(np.arange(950, 1101, 50))

    ax.tick_params(axis='both', which='major', labelsize=10) # Adjust 10 as needed

    # elev = vis_surf.get('surface-elevation', vis.cycles[step])
    # pd = vis_surf.get('surface-ponded_depth', vis.cycles[step])
    # wt = vis_surf.get('surface-water_table_depth', vis.cycles[step])
    # pd_smooth = savgol_filter(pd, window_length=11, polyorder=2)
    # wt_smooth = savgol_filter(wt, window_length=11, polyorder=2)
    # axpd = ax.plot(vis_surf.centroids[:,0], elev+pd_smooth-wt_smooth, 'black', linewidth=1.0)

    #v1 = vis.get('darcy_velocity.0', vis.cycles[step])
    #v3 = vis.get('darcy_velocity.2', vis.cycles[step])
    #axes.quiver(x, z, v1, v3, width=0.0008,color='w')
    
    if showtime:
        # days = step // 24  # calculate the number of days
        # time = step % 24  # calculate the time within a day
        # time_str = f'{time:02d}:00'  # format the time as 'hh:00'
        # ax.text(0.8, 0.95, 'Day {} {}'.format(days, time_str), transform=ax.transAxes, fontsize=20,
        #         fontweight='bold', va='top', ha='left')
        days = step
        ax.text(0.8, 0.95, 'Day {}'.format(days), transform=ax.transAxes, fontsize=12,
                fontweight='bold', va='top', ha='left')


In [ ]:
print(len(vis.cycles))
print(vis.times)

## single frame plot

In [ ]:
step=3
vis_time_value = vis.times[step]

fig = plt.figure(figsize=(14,4))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.2], height_ratios=[1, 1])
                                # width_ratios: left column width, right column width
                                # height_ratios: top row height, bottom row height

# Three plots -- first is rain and snow
ax1 = fig.add_subplot(gs[0, 0]) # Use gridspec for the first subplot
ax1.plot(time, df['rain precipitation [m d^-1]'], 'b', label='rain')
ax1.plot(time, df['snow precipitation [m d^-1]'], 'c', label='snow')
ax1.set_ylabel('precip [m d^-1]')
ax1.legend(loc='upper right') 
ax1.set_yticks(np.arange(0, 0.031, 0.01)) # Start, stop (exclusive), step
ax1.set_ylim(0, 0.035) # Ensure the y-axis limits match the desired ticks
ax1.set_xlim(time.min(), time.max())
# Add vertical line - NO LABEL HERE
ax1.axvline(x=vis_time_value, color='k', linestyle='--', linewidth=1.5)

# Three plots -- second is gross fluxes
ax2 = fig.add_subplot(gs[1, 0]) # Use gridspec for the second subplot
ax2.plot(time, gf['ET'], label='ET')
ax2.plot(time, gf['Q'], label='runoff')
ax2.plot(time, gf['F_lf'], label='flow ss_left')
ax2.plot(time, gf['F_rf'], label='flow ss_right')
ax2.legend(loc='upper right')
ax2.set_xlabel('time [d]')
ax2.set_ylabel('flux [m d^-1]')
ax2.set_yticks(np.arange(0, 0.021, 0.01)) # Start, stop (exclusive), step
ax2.set_ylim(0, 0.022) # Ensure the y-axis limits match the desired ticks
ax2.set_xlim(time.min(), time.max())
# Add vertical line - NO LABEL HERE
ax2.axvline(x=vis_time_value, color='k', linestyle='--', linewidth=1.5)

# Three plots -- third is saturation plot
ax3 = fig.add_subplot(gs[:, 1]) # Use gridspec for the third subplot
#step=3
create_tricontour_plot(step, vis, varn="saturation_liquid", log=False,
                        clim=[0,1], cmap='bwr_r', showtime=False, ax=ax3) # Pass ax3 here

# Adjust overall layout to prevent labels/titles overlapping
plt.tight_layout() # This is generally good practice

# If you had these for the tricontour plot previously,
# you would set them as axes labels for ax3, not fig.text
ax3.set_xlabel('Distance [m]')
ax3.set_ylabel('Elevation [m]')

# Adjust overall subplot parameters for alignment and spacing
plt.subplots_adjust(
    left=0.05,    # Adjust left boundary
    right=0.95,   # Adjust right boundary (giving space for colorbar)
    bottom=0.08,  # Adjust bottom boundary for x-labels
    top=0.95,     # Adjust top boundary for titles
    wspace=0.15,   # Horizontal space between columns
    hspace=0.2    # Vertical space between rows
)

## animation

In [ ]:
# --- Create output directory if it doesn't exist ---
output_folder = './images'
if not os.path.exists(output_folder):
    os.makedirs(output_folder)
    print(f"Created directory: {output_folder}")
else:
    print(f"Directory already exists: {output_folder}")


# --- Loop through steps and save figures ---
for step in range(0, 366): # Loop from 0 up to and including 365
    print(f"Generating figure for step {step}...")

    # Ensure the step is a valid index for vis.times
    if step >= len(vis.times):
        print(f"Warning: step {step} is out of bounds for vis.times. Skipping.")
        continue

    vis_time_value = vis.times[step]

    # Create a new figure and axes for each step to ensure a clean plot
    fig = plt.figure(figsize=(14,4))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.2], height_ratios=[1, 1])

    # Three plots -- first is rain and snow
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(time, df['rain precipitation [m d^-1]'], 'b', label='rain')
    ax1.plot(time, df['snow precipitation [m d^-1]'], 'c', label='snow')
    ax1.set_ylabel('precip [m d^-1]')
    ax1.legend(loc='upper right')
    ax1.set_yticks(np.arange(0, 0.031, 0.01))
    ax1.set_ylim(0, 0.035)
    ax1.set_xlim(time.min(), time.max())
    ax1.axvline(x=vis_time_value, color='k', linestyle='--', linewidth=1.5)
    ax1.set_title(f'Time: {vis_time_value:.1f} d', fontsize=10, loc='right') # Add time indicator

    # Three plots -- second is gross fluxes
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(time, gf['ET'], label='ET')
    ax2.plot(time, gf['Q'], label='runoff')
    ax2.plot(time, gf['F_lf'], label='flow ss_left')
    ax2.plot(time, gf['F_rf'], label='flow ss_right')
    ax2.legend(loc='upper right')
    ax2.set_xlabel('time [d]')
    ax2.set_ylabel('flux [m d^-1]')
    ax2.set_yticks(np.arange(0, 0.021, 0.01))
    ax2.set_ylim(0, 0.022)
    ax2.set_xlim(time.min(), time.max())
    ax2.axvline(x=vis_time_value, color='k', linestyle='--', linewidth=1.5)

    # Three plots -- third is saturation plot
    ax3 = fig.add_subplot(gs[:, 1])
    # Pass the current `step` (index) to create_tricontour_plot
    create_tricontour_plot(step, vis, varn="saturation_liquid", log=False,
                            clim=[0,1], cmap='bwr_r', showtime=False, ax=ax3)

    ax3.set_xlabel('Distance [m]')
    ax3.set_ylabel('Elevation [m]')

    # Adjust overall subplot parameters for alignment and spacing
    plt.subplots_adjust(
        left=0.05,
        right=0.95,
        bottom=0.08,
        top=0.95,
        wspace=0.15,
        hspace=0.2
    )
    # plt.tight_layout() # tight_layout can sometimes override subplots_adjust; use one or the other carefully.
                        # Since we have precise subplots_adjust, let's rely on that.

    # Save the figure
    # Use zero-padding for the filename to ensure correct sorting for animation
    filename = os.path.join(output_folder, f'frame_{step:04d}.png') # e.g., frame_0000.png, frame_0001.png
    plt.savefig(filename, dpi=150, bbox_inches='tight') # dpi can be adjusted for quality
    plt.close(fig) # Close the figure to free up memory

print(f"Generated {step + 1} figures in '{output_folder}' directory.")
print("You can now use a tool like FFmpeg to create an animation:")
print(f"ffmpeg -framerate 10 -i {output_folder}/frame_%04d.png -c:v libx264 -r 30 -pix_fmt yuv420p -vf \"scale=iw:ih-mod(ih\,2)\" output_animation.mp4")

## single 2d saturation plot

In [ ]:
step=3

fig, axes = plt.subplots(1,1,figsize=(10,4))
create_tricontour_plot(step, vis, varn="saturation_liquid", log=False,
                        clim=[0,1], cmap='bwr_r', showtime=False, ax=axes)

fig.text(0.45, 0.03, 'Distance [m]', fontsize=10, ha='center')
fig.text(0.03, 0.35, 'Elevation [m]', fontsize=10, ha='center', rotation=90)

plt.tight_layout(rect=[0.03, 0.05, 1, 1]) # Adjust bottom padding if needed

#generate_plots=False
if generate_plots:
    output_filename = './images/fig2b-massdistribution-v2.tif'
    fig.savefig(output_filename, dpi=300, pil_kwargs={'compression': 'tiff_lzw'})
    #output_filename = '../images/fig2a-darcyvel.svg'
    #fig.savefig(output_filename)

In [ ]:
# process -- snow water balance -- converting all to m/day
snowf = dict()
snowf['TF+D'] = df['snow throughfall + canopy drainage [m d^-1]']
snowf['ET'] = df['snow evaporation [m d^-1]']
snowf['SM'] = df['snowmelt [m d^-1]']

snowf_net = snowf['TF+D'] - snowf['ET'] - snowf['SM']
snow_water = df['snow water content [mol]'] / 55500 / surf_area

Note that, in this run, snow evaporation keeps up with snowfall, so there is never a snowpack!  This is a problem in the data or in the parameters, and needs to be checked.  Although there is very little snow (2cm) in total in this forcing dataset, so maybe this isn't actually a big problem.

In [ ]:
plot('snowpack', time, snowf, snow_water, snowf_net)

In [ ]:
# process -- canopy water balance -- converting all to m/day
canf = dict()
canf['I'] = df['canopy interception [m d^-1]']
canf['D'] = df['canopy drainage [m d^-1]']
canf['ET'] = df['canopy evaporation [m d^-1]']

canf_net = canf['I'] - canf['D'] - canf['ET']
can_water = df['canopy water content [mol]'] / 55500 / surf_area

In [ ]:
plot('canopy', time, canf, can_water, canf_net)

In [ ]:
# process -- surface water balance -- converting all to m/day
surff = dict()
surff['TF+D'] = df['water throughfall + canopy drainage [m d^-1]']
surff['SM'] = df['snowmelt [m d^-1]']
surff['ET'] = df['surface evaporation [m d^-1]']
surff['Q'] = df['runoff generation [mol d^-1]'] / 55000. / surf_area
surff['I'] = -df['exfiltration [mol d^-1]'] / 55000. / surf_area 

surff_net = surff['TF+D'] + surff['SM'] - surff['ET'] - surff['Q'] - surff['I']
surf_water = df['surface water content [mol]'] / 55000 / surf_area

In [ ]:
plot('surface', time, surff, surf_water, surff_net)

In [ ]:
# process -- surface water balance -- converting all to m/day
subf = dict()
subf['ET'] = df['transpiration [m d^-1]']
subf['I'] = -df['exfiltration [mol d^-1]'] / 55000. / surf_area
gf['F_lf'] = df['left face flux']/55000./surf_area #efflux
gf['F_rf'] = df['right face flux']/55000./surf_area # efflux

subf_net = subf['I'] - subf['ET'] - gf['F_lf'] - gf['F_rf']
sub_water = df['subsurface water content [mol]'] / 55000 / surf_area

In [ ]:
plot('subsurface', time, subf, sub_water, subf_net)

In [ ]:
error = gf['P'] + gf['S'] - canf['I'] - snowf['TF+D'] - surff['TF+D'] + canf['D']

fig = plt.figure(figsize=(10,7))
ax = fig.add_subplot(111)
ax.plot(time, error, '-.', color='grey', label='error in rain/snow allocation')
ax.set_xlabel('time [d]')
ax.set_ylabel('error [m s^-1]')
plt.show()
